# Integer Programming with AMPL in Python
## Carpenter Problem: AMPL Modeling for Small MILP Examples

This notebook solves the same carpenter problem with AMPL from Python using `amplpy`.

We will solve:
- the original 2-product problem
- the extended 3-product problem

Each run reports:
- the best solution found
- the profit
- the execution time

## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.

In [20]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [21]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    """Decorator to measure execution time."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
def create_ampl_instance(solver: str = "highs"):
    # Create a fresh AMPL session so each experiment is independent.
    return ampl_notebook(modules=[solver], license_uuid="default")


def extract_solution(variable):
    # Convert AMPL variable values into a plain Python dictionary.
    solution = {}
    for key, value in variable.get_values().to_dict().items():
        name = key if isinstance(key, str) else key[0]
        solution[name] = int(round(value))
    return solution


@timed
def solve_carpenter_problem_with_ampl(products, resources, solver: str = "highs"):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        set PRODUCTS;
        set RESOURCES;

        param profit {PRODUCTS} >= 0;
        param availability {RESOURCES} >= 0;
        param usage {PRODUCTS, RESOURCES} >= 0;

        var Make {p in PRODUCTS} integer >= 0;

        maximize TotalProfit:
            sum {p in PRODUCTS} profit[p] * Make[p];

        subject to ResourceLimit {r in RESOURCES}:
            sum {p in PRODUCTS} usage[p, r] * Make[p] <= availability[r];
        '''
    )

    ampl.set["PRODUCTS"] = list(products)
    ampl.set["RESOURCES"] = list(resources)

    ampl.param["profit"] = {
        product: data["profit"]
        for product, data in products.items()
    }
    ampl.param["availability"] = resources
    ampl.param["usage"] = {
        (product, resource): products[product]["usage"].get(resource, 0)
        for product in products
        for resource in resources
    }

    ampl.solve(solver=solver)

    solution = extract_solution(ampl.get_variable("Make"))
    solution["profit"] = int(round(ampl.get_objective("TotalProfit").value()))
    return solution

## Problem 1: Two Products

Variables:
- rockers
- stools

Objective:
- maximize profit

Resources:
- small parts = 54
- medium parts = 30
- large parts = 30

In [23]:
CARPENTER_RESOURCES = {
    "small": 54,
    "medium": 30,
    "large": 30,
}

TWO_PRODUCT_DATA = {
    "rockers": {
        "profit": 70,
        "usage": {
            "small": 18,
            "medium": 9,
            "large": 2,
        },
    },
    "stools": {
        "profit": 15,
        "usage": {
            "small": 2,
            "medium": 2,
            "large": 1,
        },
    },
}

In [24]:
two_product_result, two_product_time = solve_carpenter_problem_with_ampl(
    products=TWO_PRODUCT_DATA,
    resources=CARPENTER_RESOURCES,
)

print("=== TWO-PRODUCT RESULTS WITH AMPL ===")
print(f"Solution -> {two_product_result}")
print(f"Time     -> {two_product_time:.8f} seconds")

AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 230
0 simplex iterations
1 branching nodes
=== TWO-PRODUCT RESULTS WITH AMPL ===
Solution -> {'rockers': 2, 'stools': 6, 'profit': 230}
Time     -> 1.57761204 seconds


## Problem 2: Three Products

Variables:
- rockers
- stools
- new_items

Objective:
- maximize profit

Resources:
- small parts = 54
- medium parts = 30
- large parts = 30

In [25]:
THREE_PRODUCT_DATA = {
    "rockers": {
        "profit": 70,
        "usage": {
            "small": 18,
            "medium": 9,
            "large": 2,
        },
    },
    "stools": {
        "profit": 15,
        "usage": {
            "small": 2,
            "medium": 2,
            "large": 1,
        },
    },
    "new_items": {
        "profit": 45,
        "usage": {
            "small": 4,
            "medium": 4,
            "large": 0,
        },
    },
}

In [26]:
three_product_result, three_product_time = solve_carpenter_problem_with_ampl(
    products=THREE_PRODUCT_DATA,
    resources=CARPENTER_RESOURCES,
)

print("=== THREE-PRODUCT RESULTS WITH AMPL ===")
print(f"Solution -> {three_product_result}")
print(f"Time     -> {three_product_time:.8f} seconds")

AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 330
1 simplex iterations
1 branching nodes
=== THREE-PRODUCT RESULTS WITH AMPL ===
Solution -> {'new_items': 7, 'rockers': 0, 'stools': 1, 'profit': 330}
Time     -> 1.56768021 seconds


In [27]:
print("=== FINAL COMPARISON ===")
print()
print("TWO PRODUCTS")
print(f"Solution -> {two_product_result}")
print(f"Time     -> {two_product_time:.8f} seconds")
print()
print("THREE PRODUCTS")
print(f"Solution -> {three_product_result}")
print(f"Time     -> {three_product_time:.8f} seconds")

=== FINAL COMPARISON ===

TWO PRODUCTS
Solution -> {'rockers': 2, 'stools': 6, 'profit': 230}
Time     -> 1.57761204 seconds

THREE PRODUCTS
Solution -> {'new_items': 7, 'rockers': 0, 'stools': 1, 'profit': 330}
Time     -> 1.56768021 seconds
